# Synthea Data Profile

This notebook profiles local Synthea CSV files before cohort logic or modeling is written. In retrospective healthcare analytics, analysts first need to understand which raw EHR-style extracts are available, how encounters are classified, whether dates are usable, and where missingness may affect cohort or outcome logic.

No real patient data should be used in this project. Place Synthea synthetic CSV files in `data/raw/` before running this notebook.

## Why Profiling Comes Before Cohort Logic

The analytic plan depends on identifying eligible inpatient encounters, index discharge dates, outpatient follow-up timing, ED revisits, and 30-day inpatient readmissions. Those definitions should be implemented only after confirming the available Synthea tables, column names, encounter class values, and date fields in the actual export.

## Load Profiling Utilities

The notebook reuses the command-line profiling functions in `scripts/profile_synthea_data.py` so the notebook and script stay aligned.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "scripts"))

from profile_synthea_data import (  # noqa: E402
    CORE_TABLES,
    EXPECTED_FILES,
    REQUIRED_FILES,
    load_available_tables,
    print_file_inventory,
    print_table_profiles,
    summarize_date_fields,
    summarize_missingness,
    write_profile_summary,
)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

## File Inventory

This section checks whether common Synthea CSV exports are present. Optional files may be missing depending on how Synthea was generated. Missing optional files should not stop profiling, but missing patient or encounter files will limit the ability to build the core cohort.

In [ ]:
print_file_inventory(RAW_DIR)

## Load Available Tables

Available tables are loaded from `data/raw/`. Column names are standardized by the profiling utility so downstream review can use consistent names.

In [ ]:
tables = load_available_tables(RAW_DIR)
print(f"Loaded {len(tables)} table(s): {list(tables)}")

## Table Shapes, Columns, and Core Previews

Table shapes and column lists help confirm whether the raw files contain the fields needed for the data dictionary and analytic plan. Previewing core tables supports early review of identifiers, date fields, encounter fields, and condition coding before SQL joins or derivations are written.

In [ ]:
print_table_profiles(tables)

## Date Fields and Missingness

Date field profiling is essential because cohort eligibility, index discharge, outpatient follow-up windows, ED revisit timing, and readmission outcomes all depend on valid temporal sequencing. Missingness assessment identifies variables that may be required, optional, excluded, or reported as limitations.

In [ ]:
missingness_rows = []
date_rows = []
for table_name, df in tables.items():
    missingness_rows.extend(summarize_missingness(df, table_name))
    date_rows.extend(summarize_date_fields(df, table_name))

if date_rows:
    display(date_rows)
else:
    print("No date fields summarized because no expected CSV files were loaded.")

if missingness_rows:
    display(missingness_rows[:50])
else:
    print("No missingness summarized because no expected CSV files were loaded.")

## Save Profiling Summary

When local Synthea data are available, this notebook writes a compact profiling summary to `outputs/data_profile_summary.csv`. Generated outputs are reproducible from local runs and are not committed by default.

In [ ]:
write_profile_summary(tables, OUTPUT_DIR)